# Titanic survival — everything in one notebook

Step ① of the talk: data → feature engineering → model training → scoring, all in a single notebook, run by hand.

In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

## Load data

In [ ]:
df = pd.read_csv("data/titanic.csv")
print(f"{len(df)} passengers")
df.head()

## Feature engineering

In [ ]:
def engineer_features(df):
    # Turn raw Titanic rows into model-ready numeric features.
    out = pd.DataFrame()
    out["Pclass"] = df["Pclass"]
    out["Sex"] = (df["Sex"] == "female").astype(int)
    out["Age"] = df["Age"].fillna(df["Age"].median())
    out["Fare"] = df["Fare"].fillna(df["Fare"].median())
    out["SibSp"] = df["SibSp"]
    out["Parch"] = df["Parch"]
    embarked = pd.get_dummies(df["Embarked"].fillna("S"), prefix="Embarked")
    for col in ["Embarked_C", "Embarked_Q", "Embarked_S"]:
        out[col] = embarked[col].astype(int) if col in embarked else 0
    return out

In [ ]:
X = engineer_features(df)
y = df["Survived"]
X.head()

## Model training

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

## Scoring / model serving

In [ ]:
accuracy = accuracy_score(y_test, model.predict(X_test))
print(f"Holdout accuracy: {accuracy:.3f}")

In [ ]:
predictions = pd.DataFrame({
    "PassengerId": df.loc[X_test.index, "PassengerId"],
    "SurvivalProbability": model.predict_proba(X_test)[:, 1].round(3),
    "PredictedSurvived": model.predict(X_test),
})
predictions.to_csv("predictions.csv", index=False)
predictions.head(10)